In [32]:
import pandas as pd

import pandas as pd

X = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\GCN細胞分群\cell_pca_features.csv", index_col=0)

labels = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\GCN細胞分群\cell_labels.csv", index_col=0)

mapping = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\GCN細胞分群\cell_id_mapping.csv", index_col=0)

graph = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\GCN細胞分群\cell_graph_edges.csv", index_col=0)

In [33]:
print(X.iloc[:,0:4].head()) # 有到PCA50
print("----------------------------------------------")
print(labels.head())
print("----------------------------------------------")
print(mapping.head())
print("----------------------------------------------")
print(graph.head())

                                         PC1        PC2       PC3       PC4
COV19_CH11283376_CTGCCTAAGTGGCACA -15.991644  -1.202101 -1.273540  3.328791
COV19_CH11283392_ATTACTCTCCTTTCTC -12.178344  -1.200317  0.891533 -0.869063
COV19_CH11283370_CCATTCGTCGAGGTAG -11.710280   0.994717 -5.712702  6.041938
COV19_CH11931807_CCTCAGTGTTGTGGCC  -3.210713   8.520339  6.417085 -9.205567
COV19_CH11931802_GCACTCTAGTCTCCTC  -4.462946  11.155868  2.259760 -7.279695
----------------------------------------------
                                  true_label cell_type_masked  is_labeled
COV19_CH11283376_CTGCCTAAGTGGCACA   Ciliated         Ciliated        True
COV19_CH11283392_ATTACTCTCCTTTCTC   Ciliated         Ciliated        True
COV19_CH11283370_CCATTCGTCGAGGTAG   Ciliated         Ciliated        True
COV19_CH11931807_CCTCAGTGTTGTGGCC   Ciliated         Ciliated        True
COV19_CH11931802_GCACTCTAGTCTCCTC   Ciliated         Ciliated        True
----------------------------------------------
      

In [34]:
assert all(
    X.index[i] == mapping[mapping.node_id==i].index[0]
    for i in range(len(X))
)

print("Node-cell mapping is OK")

Node-cell mapping is OK


In [35]:
print(graph.shape)

(70320, 1)


In [36]:
import scipy.sparse as sp
import numpy as np


N = X.shape[0]


A = sp.coo_matrix(
    (
        np.ones(len(graph)),
        (
            graph.index,
            graph["target"]
        )
    ),
    shape=(N,N)
)

In [37]:
print(A.shape)

(3531, 3531)


In [38]:
print(A.shape)
print(A.nnz)
print((A != A.T).nnz)

(3531, 3531)
70320
0


In [39]:
I = sp.eye(N)

A_hat = A + I

In [41]:
degree = np.array(A_hat.sum(axis=1)).flatten()


D_inv_sqrt = np.power(
    degree,
    -0.5
)

D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0


D_inv_sqrt = sp.diags(D_inv_sqrt)


A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

print(A_norm.shape)
print(A_norm.nnz)

(3531, 3531)
73851


In [43]:
import torch

features = torch.FloatTensor(
    X.values
)

print(features.shape)

torch.Size([3531, 50])


In [44]:
adj = torch.FloatTensor(
    A_norm.toarray()
)

print(adj.shape)

torch.Size([3531, 3531])


In [46]:
from sklearn.preprocessing import LabelEncoder


encoder = LabelEncoder()

y = encoder.fit_transform(
    labels["true_label"]
)


y = torch.LongTensor(y)


print(y[:10])
print(encoder.classes_)

tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3])
['AS-DC' 'B' 'Basal' 'Ciliated' 'Club' 'Deutorosomal' 'Ductal' 'Goblet'
 'Hillock' 'Ionocyte' 'Langerhans' 'Macrophage' 'Mast' 'Melanocyte'
 'Monocyte' 'NK' 'Secretory' 'Squamous' 'T CD4' 'T CD8' 'T G/D' 'T MAI'
 'T Reg' 'cDC Activated' 'cDC1' 'cDC2' 'pDC']


In [47]:
train_mask = torch.BoolTensor(
    labels["is_labeled"].values
)

print(train_mask.sum())

tensor(2515)


In [48]:
import torch.nn as nn
import torch.nn.functional as F


class GCN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            output_dim
        )


    def forward(self, x, adj):

        # graph convolution 1
        h = torch.matmul(
            adj,
            x
        )

        h = self.fc1(h)

        h = F.relu(h)


        # graph convolution 2
        h = torch.matmul(
            adj,
            h
        )

        h = self.fc2(h)


        return h

In [49]:
num_classes = len(
    encoder.classes_
)


model = GCN(
    input_dim=50,
    hidden_dim=64,
    output_dim=num_classes
)

In [50]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


criterion = nn.CrossEntropyLoss()

In [51]:
for epoch in range(200):

    model.train()

    optimizer.zero_grad()


    output = model(
        features,
        adj
    )


    loss = criterion(
        output[train_mask],
        y[train_mask]
    )


    loss.backward()

    optimizer.step()


    if epoch % 20 == 0:
        print(
            epoch,
            loss.item()
        )

0 3.3392767906188965
20 0.4043923616409302
40 0.3058113753795624
60 0.2688100039958954
80 0.24376337230205536
100 0.22389821708202362
120 0.20681606233119965
140 0.191425621509552
160 0.17738628387451172
180 0.16567130386829376


In [52]:
model.eval()

with torch.no_grad():

    output = model(
        features,
        adj
    )


pred = output.argmax(dim=1)

In [53]:
pred_celltype = encoder.inverse_transform(
    pred.numpy()
)

In [54]:
from sklearn.metrics import accuracy_score


model.eval()

with torch.no_grad():
    output = model(features, adj)


pred = output.argmax(dim=1)


acc = accuracy_score(
    y[train_mask],
    pred[train_mask]
)


print(acc)

0.9391650099403579


In [55]:
from sklearn.model_selection import train_test_split


labeled_idx = np.where(
    labels.is_labeled.values
)[0]


train_idx, test_idx = train_test_split(
    labeled_idx,
    test_size=0.2,
    random_state=42,
    stratify=y[labeled_idx]
)


train_mask = torch.zeros(
    len(X),
    dtype=torch.bool
)

test_mask = torch.zeros(
    len(X),
    dtype=torch.bool
)


train_mask[train_idx] = True
test_mask[test_idx] = True

In [56]:
output[train_mask]

tensor([[ -8.9798, -13.4580, -12.8327,  ..., -11.4241, -16.8420,  -7.0624],
        [-10.9308, -16.3773, -14.2385,  ..., -13.3760, -21.2018, -10.5114],
        [-13.0450, -13.5703,  -4.9809,  ..., -11.5281, -21.9981,  -7.1833],
        ...,
        [ -7.1664,  -3.4401,   1.5166,  ...,  -1.9288,   2.9968,  -0.5880],
        [ -7.6867,  -5.0943,   3.6158,  ...,  -7.6867, -10.4183,  -3.4712],
        [ -5.9514,  -4.5373,   7.7317,  ...,  -5.6766,  -0.5457,  -1.4023]])

In [57]:
pred[test_mask]

tensor([ 3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,
         3,  3, 19, 19, 21, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19, 19,  7,  4,  7,  4,  7,  7,  7,  7,  7,  7, 16,  7,  7,  4,
         7,  6, 16,  7,  7,  7, 16, 16, 16,  4, 16,  7, 16, 16,  4, 16, 16, 16,
        16, 16, 16,  4,  7,  7, 16,  7,  2,  2,  2,  2,  2,  2,  7,  2,  2,  2,
         2,  2,  2,  2,  2,  2,  2,  2,  2,  2, 11, 11, 11, 11, 11, 11, 11, 11,
        10, 11, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 18, 18, 18, 18, 22, 18,
        18, 18, 18, 18, 18, 21, 22, 18, 18, 20, 22, 18, 18, 18,  9,  9,  9,  9,
         9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  4,  4,
         4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  6,  4,  7,  4,  4,  4,
         5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
         5,  5, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
        15, 15, 15, 15, 21, 21, 21, 21, 

In [58]:
labels["is_labeled"]

COV19_CH11283376_CTGCCTAAGTGGCACA     True
COV19_CH11283392_ATTACTCTCCTTTCTC     True
COV19_CH11283370_CCATTCGTCGAGGTAG     True
COV19_CH11931807_CCTCAGTGTTGTGGCC     True
COV19_CH11931802_GCACTCTAGTCTCCTC     True
                                     ...  
COV19_CH11931803_GCATACACAGACAAAT    False
COV19_CH11931803_ATTATCCGTTTAGCTG    False
COV19_CH11931803_CTCAGAAGTAAAGGAG    False
COV19_CH11931803_CTGAAACAGATAGTCA    False
COV19_CH11283385_GATCGCGTCTCAAACG    False
Name: is_labeled, Length: 3531, dtype: bool

In [59]:
unknown_idx = ~labels.is_labeled.values


prediction_table = pd.DataFrame({

    "cell_id": X.index,

    "prediction": pred_celltype

})


prediction_table[
    unknown_idx
]

,cell_id,prediction
2515,COV19_CH11283372_AGTCTTTTCCAGTATG,Ciliated
2516,COV19_CH11931802_ACAGCTAAGAGCTGCA,Ciliated
2517,COV19_CH11931802_CTCGTCATCCTGTAGA,Ciliated
2518,COV19_CH11931808_CTGCCTAGTAAAGGAG,Ciliated
2519,COV19_CH11283362_TCTTTCCTCGAGAGCA,Ciliated
...,...,...
3526,COV19_CH11931803_GCATACACAGACAAAT,Melanocyte
3527,COV19_CH11931803_ATTATCCGTTTAGCTG,Melanocyte
3528,COV19_CH11931803_CTCAGAAGTAAAGGAG,Melanocyte
3529,COV19_CH11931803_CTGAAACAGATAGTCA,Melanocyte
